In [1]:
import os
os.environ["HF_HOME"] = "/data/cat/ws/thha024h-masterthesis/hf_models"
import json
import re
import pandas as pd
from pathlib import Path
from vllm import LLM, SamplingParams

In [2]:
#DATA   = "data/bt_follow_2022-02-07_2022-02-14_reftweets_dedup.csv"
#DATA   = "data/bt_follow_2022-02-07_2022-02-14_tweets.csv"
DATA    = "data/bt_track_2022-02-07_2022-02-14_reftweets_dedup.csv"
#DATA    = "data/bt_track_2022-02-07_2022-02-14_tweets.csv"
PROMPT_PATH = "prompt-populism.md"
MODEL  = "hf_models/Qwen3-235B-A22B-Instruct-2507-FP8/"

In [3]:
d = pd.read_csv(DATA)

# WHEN TESTING THE SCRIPT
#d = d.head(50)


with open(PROMPT_PATH, "r") as f:
    PROMPT = f.read()

In [4]:
llm_kwargs = dict(
    model=MODEL,
    tensor_parallel_size=4
)

sampling_params = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=4048
)
llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()

to_annotate = d[["id", "text"]].reset_index(drop=True)

INFO 03-04 09:14:09 [utils.py:223] non-default args: {'tensor_parallel_size': 4, 'disable_log_stats': True, 'model': 'hf_models/Qwen3-235B-A22B-Instruct-2507-FP8/'}
INFO 03-04 09:14:09 [model.py:529] Resolved architecture: Qwen3MoeForCausalLM
INFO 03-04 09:14:09 [model.py:1549] Using max model len 262144
INFO 03-04 09:14:23 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-04 09:14:23 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=3461255) INFO 03-04 09:14:24 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='hf_models/Qwen3-235B-A22B-Instruct-2507-FP8/', speculative_config=None, tokenizer='hf_models/Qwen3-235B-A22B-Instruct-2507-FP8/', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=4, pipeline_parallel_size=1, data_parallel_size=1, disable_

Loading safetensors checkpoint shards:   0% Completed | 0/24 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:16:22 [default_loader.py:293] Loading weights took 98.24 seconds
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:16:22 [fp8.py:495] Using MoEPrepareAndFinalizeNoEP
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:16:23 [gpu_model_runner.py:4221] Model loading took 55.19 GiB memory and 106.881191 seconds
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:16:47 [backends.py:916] Using cache directory: /home/thha024h/.cache/vllm/torch_compile_cache/62d59a945c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:16:47 [backends.py:976] Dynamo bytecode transform time: 23.90 s
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) WARNING 03-04 09:16:58 [fused_moe.py:1087] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/cat/ws/thha024h-masterthesis/venv/lib/python3.11/sit

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|█| 51/51 [0
Capturing CUDA graphs (decode, FULL):  98%|▉| 50/51 [00:10<00:00,  4.71it

(EngineCore_DP0 pid=3461255) (Worker_TP1 pid=3461271) INFO 03-04 09:17:47 [custom_all_reduce.py:216] Registering 19278 cuda graph addresses


Capturing CUDA graphs (decode, FULL): 100%|█| 51/51 [00:10<00:00,  4.82it


(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:17:47 [custom_all_reduce.py:216] Registering 19278 cuda graph addresses
(EngineCore_DP0 pid=3461255) (Worker_TP2 pid=3461273) INFO 03-04 09:17:47 [custom_all_reduce.py:216] Registering 19278 cuda graph addresses
(EngineCore_DP0 pid=3461255) (Worker_TP3 pid=3461275) INFO 03-04 09:17:48 [custom_all_reduce.py:216] Registering 19278 cuda graph addresses
(EngineCore_DP0 pid=3461255) (Worker_TP0 pid=3461269) INFO 03-04 09:17:49 [gpu_model_runner.py:5246] Graph capturing finished in 24 secs, took 1.32 GiB
(EngineCore_DP0 pid=3461255) INFO 03-04 09:17:49 [core.py:278] init engine (profile, create kv cache, warmup model) took 86.04 seconds
INFO 03-04 09:17:50 [llm.py:355] Supported tasks: ['generate']


In [5]:
# PROCESS TEXT
prompts_list = []
rows_list = []
for row in to_annotate.itertuples():
    conversation = [
        {"role": "system", "content": PROMPT},
        {"role": "user",   "content": row.text},
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts_list.append(formatted_prompt)
    rows_list.append(row)

print(f"Starting inference on {len(prompts_list)} items...")
outputs = llm.generate(prompts_list, sampling_params)

Starting inference on 55861 items...


Adding requests:   0%|          | 0/55861 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/55861 [00:00<?, ?it/s, est. speed input: 0.0

In [6]:
STR_KEYS = [
    "holistic_redescription",
    "social_actors_analysis",
    "populist_explanation",
    "elitist_explanation",
    "intensity_explanation",
]
NUM_KEYS = [
    "populist_score",
    "elitist_score",
    "intensity_score",
]
d_results = []
for row, output in zip(rows_list, outputs):
    answer = output.outputs[0].text

    # --- 1. Attempt Standard JSON Parsing ---
    try:
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback ---
        try:
            parsed = {}

            for key in STR_KEYS:
                match = re.search(
                    rf'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)',
                    answer,
                    re.DOTALL,
                )
                if match:
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            for key in NUM_KEYS:
                match = re.search(rf'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None

            answer_data = parsed
        except Exception as regex_error:
            answer_data = {
                "error": f"Parse Failed: {regex_error}",
                "raw_output": answer,
            }

    d_results.append({"id": row.id, **answer_data})

In [7]:
# Convert annotations to DataFrame and left-join to original
d_annotations = pd.DataFrame(d_results)
d_out = d.merge(d_annotations, on="id", how="left")

In [8]:
d_annotations.head()

,id,holistic_redescription,actor_analysis,people_explanation,people_score,elite_explanation,elite_score,antagonism_explanation,antagonism_score,social_actors_analysis,populist_explanation,elitist_explanation,intensity_explanation,populist_score,elitist_score,intensity_score,holistic_redplanation,holistic_reddescription
0,1490458710036123649,No semantic argument detected.,The text mentions two individuals: @n_roettgen...,No broad ordinary majority is invoked. The tex...,0.0,No generalized elite is targeted. While @n_roe...,0.0,No opposition between 'The People' and 'The El...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1490460164692054017,No semantic argument detected.,"One individual is mentioned ('n_roettgen'), a ...",No broad ordinary majority is invoked. The pos...,0.0,"Although the addressee is a politician, the po...",0.0,No opposition between social or political grou...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1490418959778304001,No semantic argument detected.,One individual is mentioned ('eine aus der DDR...,No broad ordinary majority is invoked. The ref...,0.0,No powerful group or class is targeted. The co...,0.0,No opposition between 'the people' and 'the el...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1490433699011338246,No semantic argument detected.,The text refers to a physical structure (the A...,No broad ordinary majority is invoked. The pos...,0.0,No powerful group or class is referenced or cr...,0.0,No opposition between social or political grou...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1490419588080799744,The post expresses personal frustration and an...,1. 'This pack' / 'this rabble' ('dieses Pack' ...,The speaker identifies personally as coming fr...,0.0,The terms 'this pack' and 'this rabble' are us...,-1.0,There is a moralized opposition between the sp...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Save as parquet: derive name from DATA
stem = Path(DATA).stem  # e.g. "bt_follow_2022-02-07_2022-02-14_tweets"
out_path = Path(DATA).parent / f"{stem}_annotated_populism.parquet"
d_out.to_parquet(out_path, index=False)
print(f"Saved {len(d_out)} rows to {out_path}")

Saved 55861 rows to data/bt_track_2022-02-07_2022-02-14_reftweets_dedup_annotated_populism.parquet
